# Convert SDFs into `SDFdataset` as a member of the `GNNdataset` that inherits PyTorch Geometric `Dataset` for data handling methods 

## Load sdf and test feature extraction

In [ ]:
import sys
sys.path.append("..")  # Ensure notebook can find the package

from rdkit import Chem
from atoMLtype.RF.featurizer import GraphFeaturizer
import importlib

# Load the SDF file
sdf_path = "./data/parm_at_Frosst/zinc.sdf"
json_labels = "./data/antechamber/atomLabels_gaff2.json"
suppl = Chem.SDMolSupplier(sdf_path, removeHs=False)

# Initialize Featurizer
featurizer = GraphFeaturizer()

# Extract Features from First Molecule
mol = suppl[0]  # Get first molecule
if mol is not None:
    print(f"mol: {mol.GetProp('_Name')}")
    atom_features, edge_indices, bond_features = featurizer.featurize_molecule(mol)

    print("Atom Features (Node Features):")
    for i, f in enumerate(atom_features):
        print(f"Atom {i}: {f}")

    print("\nBond Features (Edge Features):")
    for i, (idx, f) in enumerate(zip(edge_indices, bond_features)):
        print(f"Bond {idx}: {f}")



## Test GNNdataset


In [ ]:
import atoMLtype
from atoMLtype.RF.GNNdataset import GNNdataset
from torch_geometric.loader import DataLoader

# Initialize Dataset
dataset = GNNdataset(sdf_path,json_labels)

# Check First Graph
graph = dataset[0]
print("Graph Representation:")
print(graph)

print("\nNode Features Shape:", graph.x.shape)  # (num_nodes, num_features)
print("Edge Index Shape:", graph.edge_index.shape)  # (2, num_edges)
print("Edge Features Shape:", graph.edge_attr.shape)  # (num_edges, edge_feature_dim)


## Test data handling of PyG dataset that is inherited in GNNdataset

In [ ]:
# Use PyTorch Geometric DataLoader
dataloader = DataLoader(dataset, batch_size=5, shuffle=True)

# Check one batch
for batch in dataloader:
    print("\nBatched Graph:")
    print(batch)
    print("\nBatched Node Feature Shape:", batch.x.shape)
    print("Batched Edge Index Shape:", batch.edge_index.shape)
    print("Batched Edge Feature Shape:", batch.edge_attr.shape)
    break  # Only print one batch


In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
import torch
from rdkit import Chem
from rdkit.Chem import Draw
from torch_geometric.utils import to_networkx

def visualize_molecular_graph(data, mol=None, atom_labels=None):
    """
    Visualizes the RDKit molecular structure next to its PyG graph representation.

    Args:
        data (torch_geometric.data.Data): PyG molecular graph.
        mol (rdkit.Chem.Mol, optional): RDKit molecule object for reference.
        atom_labels (dict, optional): Dictionary mapping atom indices to labels.
    """
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))

    # ---- Left Side: RDKit Molecular Structure ----
    if mol:
        atom_indices = [str(atom.GetIdx()) for atom in mol.GetAtoms()]  # Get atom indices
        img = Draw.MolToImage(mol, size=(400, 400), legend="Molecular Structure")
        axes[0].imshow(img)
        axes[0].axis('off')
        axes[0].set_title("RDKit Molecular Structure")

    # ---- Right Side: PyG Graph Representation ----
    G = to_networkx(data, to_undirected=True)

    # Assign colors based on atom types
    node_colors = [data.y[i].item() if data.y is not None else 0 for i in range(data.num_nodes)]

    # Layout for better visualization
    pos = nx.spring_layout(G, seed=42)
    
    nx.draw(G, pos, with_labels=True, node_color=node_colors, cmap=plt.cm.viridis, 
            node_size=500, font_size=8, edge_color='gray', ax=axes[1])
    
    axes[1].set_title("PyG Graph Representation")
    
    plt.show()



# Get a sample graph
data = dataset[0] 
mol_name = list(dataset.sdf_dataset.X_molecules.keys())[0]  # Get the molecule name
mol = dataset.sdf_dataset.X_molecules[mol_name]  # Retrieve RDKit Mol object

# Visualize
visualize_molecular_graph(data, mol)



# Choosing and building a GNN

For a basic model, start with Graph Convolutional Network (GCN) or Graph Attention Network (GAT). You can later explore Message Passing Neural Networks (MPNNs).

In [ ]:
from atoMLtype.RF.GNNmodel import GNNTrainer, BaselineGNN
from atoMLtype.RF.GNNdataset import GNNdataset

# Load the SDF file
sdf_path = "./data/parm_at_Frosst/zinc.sdf"
json_labels = "./data/antechamber/atomLabels_gaff2.json"

# Initialize dataset and DataLoader
gnn_test_dataset = GNNdataset(sdf_path, json_labels)
num_node_feat = gnn_test_dataset[0].x.shape[1]
num_y_ATs = len(set(gnn_test_dataset.all_labels))
print(f"num_node_features = {num_node_feat}")
print(f"num_y_ATs = {num_y_ATs}")

# Initialize baselineGNN for classification
model = BaselineGNN(num_node_features=num_node_feat, num_atom_types=num_y_ATs)
trainer = GNNTrainer(model, gnn_test_dataset, batch_size=32, learning_rate=0.001, epochs=20, task="classification")

trainer.train()
trainer.evaluate()


[15:48:21] Explicit valence for atom # 9 N, 5, is greater than permitted
Sanitization failed for molecule at index 2131, Name: ZINC16448882. Skipping sanitization: Explicit valence for atom # 9 N, 5, is greater than permitted
[15:48:22] Explicit valence for atom # 7 N, 5, is greater than permitted
Sanitization failed for molecule at index 2721, Name: ZINC15772239. Skipping sanitization: Explicit valence for atom # 7 N, 5, is greater than permitted
[15:48:22] Explicit valence for atom # 10 N, 5, is greater than permitted
Sanitization failed for molecule at index 3249, Name: ZINC11539132. Skipping sanitization: Explicit valence for atom # 10 N, 5, is greater than permitted
[15:48:22] Explicit valence for atom # 3 N, 5, is greater than permitted
Sanitization failed for molecule at index 5699, Name: ZINC17111082. Skipping sanitization: Explicit valence for atom # 3 N, 5, is greater than permitted
Skipping molecule ZINC59391023: Missing from JSON.
Skipping molecule ZINC00335972: Missing fro

num_node_features = 136
num_y_ATs = 75
BaselineGNN - Number of parameters: 17803
Epoch: 1/20


  0%|          | 0/200 [00:00<?, ?it/s]/Users/brobello/miniconda3/envs/OFF_playground/lib/python3.10/site-packages/torch/nn/modules/loss.py:610: UserWarning: Using a target size (torch.Size([924])) that is different to the input size (torch.Size([924, 75])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
                                       

batch_pred shape: torch.Size([924, 75])
batch_data.y shape: torch.Size([924])


RuntimeError: The size of tensor a (75) must match the size of tensor b (924) at non-singleton dimension 1

# Train & Evaluate the Model
Prepare Data for Training
Split your dataset into training, validation, and test sets.
Use DataLoader for batching.

In [ ]:
from torch_geometric.loader import DataLoader

train_loader = DataLoader(graphs[:80], batch_size=32, shuffle=True)
test_loader = DataLoader(graphs[80:], batch_size=32, shuffle=False)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = GNN(in_dim=101, hidden_dim=64, out_dim=10).to(device)  # Adjust output size
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = torch.nn.CrossEntropyLoss()

def train():
    model.train()
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        out = model(batch)
        loss = loss_fn(out, batch.y)  # Ensure 'y' contains correct labels
        loss.backward()
        optimizer.step()
    print(f"Loss: {loss.item()}")

for epoch in range(50):  # Train for 50 epochs
    train()


# What are the common fingerprints for the different atom types
1. Are the rings something to consider?
2. What about functional group as immediate substituents?
3. 